# 02. Scenario 1 — "My card doesn't swipe. What should I do?" (clarification loop)

The first support scenario. The inquiry **lacks specifics**, so the agent decides `need_clarification` at D5 (information sufficiency) and runs the recursive loop: **write a clarifying question -> receive the customer reply -> re-decide**.

Expected path:
```
D1(guardrail) -> D2(routing) -> D5(need_clarification) -> ask/reply -> D5(info_sufficient)
-> D3(tool choice) -> tool success -> D6(goal_met) -> summary -> D7(no_risk) -> D4(resolve) -> final reply
```

We run the same question once with **(1) Haiku decisions** and once with **(2) SGLang decisions**, watching the STEP output, then compare the execution times at the end. (Reply generation is Sonnet 4.5 in both arms — only the decision layer differs.)


In [ ]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import ws_common as W
graphs, client = W.make_graphs()
sc = W.scenario(1)
print("Scenario:", sc["title"])
print("Customer question:", sc["question"])
print("script (pre-set for reproducibility):", sc["script"])


## Where this scenario sits in the full workflow

The path this scenario takes (red borders): intake screening -> routing -> **clarification loop (clarify_gate <-> ask_clarify)** -> one successful tool call -> summary -> no risk -> resolve. The KYC sub-flow and the give_up path are not visited this time.

Assigned team (red box): **declined_pay (declined/pending pay)** — in English, "my card doesn't swipe / won't go through" reads as a payment being declined, so D2 routes there. (The Korean phrasing of the same scenario routes to the **cards** team in the [KR] notebook — routing is wording-sensitive, which is itself worth observing.) Compare with the actual D2 decision in the STEP output.
> Note: this diagram is not auto-generated from the running LangGraph; it is a **visualization-only drawing** rendered from the caption dictionaries in `ws_common.py`, keyed by node name (details: `[EN]_01_build_agent.ipynb`).


In [ ]:
W.draw_graph(highlight=W.scenario_path(1), show_teams=True, width=760)

## (1) Run with the Haiku 4.5 decision agent

Each STEP prints which node makes which decision (label) in how many ms, plus the clarifying question, the customer reply and the final answer. Note that **every single decision takes 600ms or more**.


In [ ]:
res_haiku = W.run_verbose(graphs["Haiku"], sc["question"], sc["script"], "Haiku")

## (2) Run with SGLang (Qwen3-VL-8B) decisions

Same graph, same question. The difference: decisions finish in **30-50ms** and come with a **confidence**.


In [ ]:
res_sglang = W.run_verbose(graphs["SGLang"], sc["question"], sc["script"], "SGLang")

## (3) Execution time comparison

Generation (Sonnet) time is similar on both sides — the point is that **the decision layer's share of time disappears almost entirely**.


In [ ]:
W.compare(res_haiku, res_sglang)

**What to look for**
- D5 runs twice because of the clarification loop — the more the loop runs, the more decisions pile up, and the bigger the cumulative gap for a slow decision backend.
- Compare whether both arms produced the same labels (rightmost column of the table). If they match, this conversation gained speed with no quality loss.
- Next: `[EN]_03_scenario2_retry.ipynb` — the loop where the agent recovers on its own when a tool fails.
